# XBRL validation (Part 11)

Loads the filing's own XBRL with Arelle, maps PDF row labels to concepts, and compares both extraction paths. Run `dvc repro xbrl` first; this notebook reads its outputs and re-runs one filing interactively.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '../src')
import xbrl
from common import XBRL, load_manifest
pd.set_option('display.max_colwidth', 80)

## 1. Non-dimensional duration facts for the latest fiscal year (Exercise 11.1)

In [ ]:
facts = pd.read_csv(XBRL / 'facts.csv', parse_dates=['end'])
tenk = [s for s in load_manifest() if '10K' in s][0]
f = facts[(facts.stem == tenk) & (~facts.instant) & (facts.dims.isna())]
latest = f.end.max()
f[(f.end == latest) & (f.days > 350)][['concept', 'label', 'value', 'decimals']].sort_values('concept')

## 2. Label -> concept mapping and method per line

In [ ]:
cmp = pd.read_csv(XBRL / 'comparison.csv')
cmp = cmp[cmp.status != 'no_table']
cmp.groupby(['path', 'statement', 'label', 'concept', 'method']).size().reset_index(name='periods')

## 3. Match rate per statement and per extraction path

In [ ]:
cmp.groupby(['stem', 'statement', 'path']).status.apply(lambda s: (s == 'match').mean()).unstack('path')

## 4. Every non-match with its suggested cause

Confirm each cause (OCR, table structure, normalization, mapping, period alignment, extension concept, rounding) and record the fix in `reports/xbrl.md`.

In [ ]:
cmp[cmp.status != 'match'][['stem', 'path', 'statement', 'label', 'period', 'raw', 'pdf_value', 'concept', 'xbrl_value', 'status', 'suggested_cause']]

## 5. Re-run one filing interactively (after changing `config/label_map.yaml`)

In [ ]:
m = load_manifest()[tenk]
fx = xbrl.load_facts(xbrl.ixbrl_file(m['accession']), tenk)
redo = pd.DataFrame(xbrl.validate(tenk, m, fx, xbrl.load_map()))
redo[redo.status != 'no_table'].groupby(['path', 'statement']).status.value_counts().unstack(fill_value=0)